# ILLM Lab 7 — Noun Phrase Tagging with a Hidden Markov Model (HMM)

## Assignment tasks
1. Use the **forward algorithm** to calculate the probability of every test sentence.
2. Use **Viterbi decoding** to predict token-level tags (`B-NP`, `I-NP`, `O`).
3. Evaluate predictions with `seqeval` when available and report precision, recall, F1, and accuracy.

**This notebook is self-contained.** The supplied `train_NP.txt` and `test_NP.txt` files are embedded in compressed form, so you do not need to upload them separately. The model is a supervised HMM estimated from the provided training annotations, with Add-\(k\) smoothing and log-space dynamic programming to avoid numerical underflow.

## 1. Imports and reproducibility

In [ ]:
import base64
import gzip
import math
import os
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 2.1.3
Pandas: 2.2.3


## 2. Load the supplied train and test data
Each non-empty line contains `TOKEN<TAB>TAG`; blank lines separate sentences. The embedded compressed text is the exact content of the supplied files.

In [ ]:
TRAIN_GZIP_B64 = """..."""
TEST_GZIP_B64 = """H4sIALdU0GoC/5VbW3fbNhJ+bv/GXvxi69ixnTiPkZO2bhM7J0nPnn0rREISapJgANKK+usXmAswoCg3+5LYFoXLzDffXHlru7WpdVfpH5Zn9x9/NN0PDz8OW/qtt2NX/3AHH/jwwS482uzDD/pbr6tB1/FhG/9Rjzr8pzobvuvwy36rXI9frs2TpmXW8WmnatphbTaj0x4/XFsXPv2s+0G3K17mNPypHuPq+LHTjVaevj7Y1jpnd/nRtTINn8pvwyfhVHSeceUH1Q1GNXSWtnf2Sbe6G2h/Z9vw/K9juCN8RfHt34yb0dNTJx4/7LRyZ05X1tFDtV6bygx0mUVY6cfbrQqibRo4uV1L2b77Vm311zFcEx6/NxsdD/5e7byNSuBtgnQGFUUNj1W2bc2Qjwz3VCxL1+JfW9vpQbk9/tbbxlT081ZFPW5102fl9U4/xQXzQk7rtWpYTPE0ftCuMd0GHwhScwVOFAtnp/WjuP9yhGU71ez9kC5UPcINA7bCovu0qh/73rpBYqHcFg+/0jp+Wztbwx1We3mUKku80FbERUCalFrXBXhXiNs963R3KLM24C2DFMSxNbTsB9V5E24DH/1iR087+F7raos/N1E68PSX7eh8rfZCQl/SUng701Uu4ruWl3LGk1RLDG2iIrqMhpWOsgJLqbJ+eckoq2Q5LmDKC2lcvMS//6swhbCTITusRufSRhfX+D893QQANWKxYAm6q48wSbTSYDWdbU3ChIKnw7G12XT4J/0tqnJD8myVe9S0eYITSlpF/YMs//gj/PtFCs0fBxNIkgC/UnQ2ZKa7zEy0o4vCC7Leg/GoJwMsaAuyrO2OcOANLxLvenIS91Ym3vB261jdb00A6S4/5nVnLMn698VvC/wkCYoOEs3p3rajo5N/0j7QECPtrvODGcZBC4CBUO4INo5p3k/Zek4E3Qg0nI4oFqjs2IDxofmQFnfrkWDQ2EEANjCMjyaUaRqEEvYHlH42gbDws2WQT2WF9EAUyxlRoE4/mLqJ4EFjtN2gNmP+Mglp7Iw8TfGdperItD6+vxVi+ylgsVIJoLhb0vUBTghoDhArHGU8xQpIcK13k3vQd9CXFkhSgxLWVLoxQGG2ifp5X5bW/Af++QURRQBfoq2z95+WZ/jjP/G/y8Xsc5/Sc+TqBCNmApY8oaoAFP7F6Ujvuk6unLzqUpAhLSL9bbGXKmmFbzV73o3qeeONicYPW8NphYsH0x5sjw7RdqC20qekXS5ndimOB/h6qAabwpeL1zc3AlafVfenoq1/tX5rMljfjc72mm1pzvCXyiX6WjowXsLQv/G/W7sQhIKUM7X6zpK3jUyLINvi6s4MytD27DZb1Y1rVQ0j7EzqroNuGISwZjCHzgedtuh+QgykmzVHaFGc1voBw8aAAC80/qYAvlcte2jTChJIlK7bcEKPYZ4DTfWBWUyUk6mijazsWKwYfm2sfZSUEUwlHyKuvgnRaQRFkPhgulETM2zNZstBV+fHlpkwBYWV6oPEiPA21tYpQlgHZpdh4C9ahj7++8ylANYuQC4qT0G0S44dwCoRerG4mUEoIH4SVeOx7E5jJBfF8MEtDj0TgYhPu9KNCV/BGygRMgbbGquh3NKpHYMoGF8Ku/1gq8fVaJo6QQrMLeGqcDAh2E+BjDpMLILJiB0L1WIAbtuV6cC0d2bYxkWIcGIWQcL1lsHm9EqkPN0Usmo96ClxsaWMXUqKSH1VCDZ04T2J3qVi+Z6lZhUt2phhaDgIIUQlXc+xEVw6afLAm+62ZP2q8ZYiLvRySbrPLn8crYlsYRvlOoERjfZl1t/jQpGQV2PY1W+Tqyhjt9NDC+JgfRev1Z0ANOsnhfF9O4YIiQBcqRFCYdMFHA/WkaTJwHYKEiH0uxqyztHlKEk/UbI8E4wHjgxKzPBZ6ShdOMefAS1Ijb31JpqJxOh9NMKwTBOojGyR1+QIKf4NArl70p9lZM2ce4NBPybmfyfuaBXImZz8NWrgHAjPSonUCTvPuaAecRe8125rGk0njtlJTCPnRFcshM5uzzZ4QPJVY0dM9mCfu/87BSv4jVJsvH5AKuVaks5Y6bhQYMMc3Feq47hXVEFUQeqBh4LPnANJk0gnmKaW1NXFAIVJRWRnIbwPnEv4G+KtOcUf6HKx+BK31+t1DCQzsHLsDHnf3WRltDQ1lHledHuaDykSO2TD8CHtsA8H8wepfnKVSw5tuzpkJJxI5ciWyjhRdj85U3NMVGfz9eNmo/2QdZOEOE2PZCgSlYWmRt/6OprqsUk5YgoxCyugEJu+4uHUTDbOAl+cLy6KDLk0haAYLgN9HZUbGDvTMNzHE889iIHtDnA09uGfy+DLZzJylaVPXwRzn9QSKhsQ6LQXbo/C0cXLInOfoHD2nNM7rI1jfMxcgTa6XlwVG6UixP8nFrAeiKGXh+5tJlCZyaGSbufIBkgVgTMgkwa8DBp+dxZwlNJVzOMIlpnDCTTZx5QWn8BE9THgaaA/r92TqbRPJr+lqprqDixJhJfhPhgyQMwFLgpMMe/dWpeJKFXUoqqL2mRwOoFoU5p6tISGhak2+o8Y29NDTURgMpmtTDFCuh/RN1eAOkpzaqNMChI6HcTiFZRbKIEAEOsOqghTtj6o7JaMXVsMWLFG3TeKi92cTKLRFxEGfQeZiEO5HAwkg6N63nImUwnxLIggGaUaa5M94JTUprU8QCY6mxAFxMsC2MU3spRIvxsrrSYVgZfTgiaKVHMJtqUAeYWVfSq7JQUxcNVjFwOUBCCwiR/fFLVdtXFazySdFMeig7EY+U9FDuXr7I8odbBZIrY0kFl/ztF472LyXIoa0SajPZBBP/otKWqDGxxgEFPVrs6WQlKN4QP9MdWQi9hhkqAc1XY6t+3HJqT5w75MD/zOrAcw1RZhPCRYFVf1EJI9RCEjWrDqUAganYxqgk3We6ovkUJiIpSqA16ASXH4UslCdDXXPfp+vUjHnwO7sg4S49Zcehg4gwKDTATzT3Zu1zevz6UzerF4ffX6mrNQl+rB6H7y115fHXztxeuDrzXRGo4V8t9NStrkJ2RwWSZAInqOzRY/yVNQfeEk5wfnOH0OSRLaR/sAwhTmqv+nmXLh461qhuMQl4kuym/K0+mwK9WoMn06zlI5Dxg75KSUAEjgnKQ0IHcI3lSDeUo2lGO+OlgFx00hu0HlbGCXjN8QAOhhBnxgNUHJlXb8PJebdAwajE5/VYM46/GmRq1Vk0p4KYApJVZmt7INwt2wzdixnmoT/WClJzW4abT1ZAOSDdcVoB7DpWh5PCwYGIYk7hd8u62qsTdlh0oezBs8wkM1EBNcXMoVdcGEbzmC+zUgwXMfA0pByZ+/iYkkH6tvRsJCjnPAJWDdggmgtyHWKOIdluTysIkUblWbSuQak5q2bUUExx3OHBCTiu+e71uRZfptoHLUEH+9Gukcq8ivpEyqAFdscl2topWrRrDAo9Y9UrfisMwyu+o92xn2H6COkkx4SvClHQaaKItzvy8+ky5/0nXM6vGz2H5yT7Ld9F4N+kB+99xW/a91j7nNlLZITFrWyYS1os/6OlI9BHR0sbi5fnUzy46QQRHsw2NXr86/m8zh5PfcrMkHLrNHkMfdTPmJfCtU1wAIqJ7wU830AGe/erG4uuRcrjty8quLBZ88PXTsfM86J3CsX+zjPpWPoqMOKZeMXYe/u1xwy51o37AK5a0uFuxuj95qjrzxaHcTBGPyF9a8nK4Jl3ooMHZr29bWifTfFWZ4TGhQ/g/xqGzycY2TIo3G5I6b1zGMTQCk+OHy5avFZYofuO+Q3Fy6/wtSZaVLYnoXsskWXAaz88j1aIITF4Nj8YVaUZRdt7I6C5v6qdpjAWk/F0zFD3+xrMDfLH8wxUUW0JwVJiG8XFwfFwJeNA1CfEou52NI9rvkBN+b1qQKOn+0NX3hIjkKDLFkcOI5vlFVcMMQdrxZfCS2+tlpxgw7lexkqoXsC9FFUiSYmw16KlK+IHg4cjtcMBtS7QIONXsUhv3KKiemIubkM61r2PWa18eyxbj6k9rHRb038ntStQzhW+02qWIT0zUxPrLH/uADkIlItbInv7yYY0N5a8iFhkFVWz1t3KxNk90olqXSeT/rYHcYRJVdttKQo5Ubn+u9ZaOTqKiPvfA05SUJGTp8G6zBjD2woF3FKgjlPyEs8rHdadkRrk0XAqp0akoEqUkdVXs3UW3Qj6lS4DJNVOL4lqgAiZJaU+6RREsd+avzM3UmsIjY523CZUftJxFLH319bh5jrCrXvDh/ydXA9qCHNItbokbIQzEcfHH68uXV6fnrG2EtPoUdMRqlXB+NJyVHOeYEvj9gk5sJd6ZU7ebVdf4IKKxTsRinmjyadhbEcAa9qWSKkvhKAReSMoJuP+hvprK8GTHjovg95uKy5OL02kn0BC3Ftqj/Gw4kdDK9HUUGx6YDswenMCbLNVYHMr3PahGQeLN4VaR8pB9uO+2whlCDji8vT8/Pz58BfYQ1o0dCAHVUNSOB4PI8LySfC9CMru3h6KhcwdCX18nTDLacWxE+aJ6+ixjgbAUdCMo+tnSB38ZtNyWhT3Ggg1jyg6reRrylicxucLY5Qt+le8sup4i3vOVcLFBTDq57/K7A13c40MXHOZDnHRrpYQ93ECOdE+cLpvEf7YedtXVqL/Mp3qd5jfKO7ClN4AGnuScWC7amNql0CVuXa//s7NinFWWfCesSR0+CFYiji1F7mVpPYCTLYEY8f0blBdHnEjXq01QLTmOWkDmxUkoJhidTyzW3nalu1vZ6wOYqNQkis6YibMiwQlrfqPwAyBKPbofB0qANZnaApdHRocZ+F4OKh1TddjrcgD4E01+PMGkcx1TEDggYN1aPos1OFm36PpdB2CaUcWdrktRdZtLkwbfqSYsx25rKTowHrEUJyfoYMYyNTDBSjTNPI8MHIqTt9DfaPrfETpGKm4A3yr6wskt8n7J+klC8ASQ2pLY2XRGPsZZwgFGn3FTBhpT606ah9yArUdPZF+XiNOq3PoIw0A/05Mgf4pQK9ZfX4ueJ9vAqrdpj3U95jN9KX5JlteaEfNrF/bJVea4lz8caUr4MxSidC1adsiWxGpV5iotuVcD0Zi61WbkMoyDdYsQzlx7InYrS3ikNswt0x+6So17SvIygfp5XLefZWpzZSBaKGF45q+pizPgtwkRUxoKFoxODgGUa38bh9kbSXYXtN0LVvteyGjMPOTUUcS/eR5RykSBCDOAGIZDWwh3gVNCzKea2UoMH+00NNrtxvAwGbyDrxM6D5Zpugaos5btyxYPh58as1/FtCQz41d6JB9E9sBmBEBq7gbk+WTd5O+IvHy0nLbdWjley+ZHM3ZPeI/MASvNs58PWUCnk85AO/HsXc32f0JUTi6n9inkJUtXZhBhOU3E1Vy4Tr8Q6rTOrMUtNsFLBATloL9vpmRoz7VFRhJnx4rz4RumcXnCcVBJWbxskYPmCz9/edXr44Pm40XBKtkTtYLKKy/Mi9kxTaVswsjiwmVOm59iXDJ7a4HiMXMjvU7Q0ZTl6UYBFaPyR8YYsW65pT2yRPIbw1jn2vnh9c15MCOy4AYqOES/rU8tHccdQtSpOs5aQAQaSAevYR58v9gtarHQAcH2cNSaxjDTOpY7TazV37t7bTr7DoI6vma2UE9O19p470XDxWWODM9/GWtdq9HMz4IdixRxLmSbKwk9e0Ro4dJmLRYbk1lZ6ozr2jFlFulMrRL4vKARb5crRdEIqfB6AAFsMyjmTvsu9Wwz8hBnFdlGfyy7EqxoqB8xSpzT7Kog8praV6lUVCYqAlRrRGtdQG6zbgOVgv2GGGKtxviJweT0z83n4KlAI6UZOUk5ZmyHgOKsSdYI2JhCOmjsgjNNUIzqsTzBHFcc5Zc8kvvByMmwEQNk46/lVPS4R3IkwQvieOCqS6tPl2BUs9fP9R8ErATUXQuk4+EbMdmwKyHSsXi6m+6+j1n/hSwewQvmFI1Tr9yGya0WYa3wK12VkC2hozCO+hZIHPigshm3R5TZ7tgYZJReBvaCNTJ8loij0UHvE6x/TVHKfCp8P/PYgMNCEhcTsgGCgY5uKWV98D2nVpOAFpv7nDfasShmbV0/hwF40eLNJEU1P3vzJA5xCQMx2XeqDH8TVSY3TkP/gElw8h5ch46tLRQ3ErtexC4FizheZ5FOgT/kurt60uetQIq2Yh8uDPVTywB6tblSn9wf+AGbUmjh6Jdj+jRu2I+37dsHlgTzHXhYIblUbgv96I94N+6C8X+RfCRqg///QYyegmk1IhSHLSEPG8VKUcAeDM8WFD5NqIzIUAYr0LoNU4D2UxNiFz0faODAe9NCUNuyE0yvURHmDdBpO44Dag/ASSRQ95JOFnY76kFMxMcCHc48c8dME2pTvgPzXUkR9gs1puEJ5pCHP4cXXnSGRyJNCk7KRM9zyCU5mKEuX+HxVca0jvnhs7OjTdC4i794OPDKu3T5Nuh++eyJmYGyd3xsqOJDIht4iKOijeGeUpXQiZ0G58fbMy4y2K3MwwBTMzixLn1cGKvNB2M/apRjsQ3U7Nk0sjOZnI3WHnZ18V38mFgHhRCtS1XaMbEFnm7weCrf5oqttZ0MQJ+ew3ocbnsU87wwO28TkdzkTVk0mRCJs9jNOTGznNbwTMFcswQFzxYsJMcV8vpH9pTq+1hLL+1Dn2gHQSVdXi1flgPOB7aUXJhtofcEaue78JiC4me0TsE3ciXiLygRYDTQxAAcsU2EQZz9m69CfKIo9O1oSe8aFY4EjlwvjbCZW+BLmYtwQAw4xxl+Qw2Os3SeVci3yWNGLQ8CokuLtvS41JUXTC+B3G+jTNqbOfW2qqgZtzVbDU9ela2yqbD8WQeatasy67MA0M0i9E+o66BJmeTOIyoZ4JqqyLjQzbQXUEXxACE/NX7qgsO+4faB5fEfKYlLjtWzkiADhenFZRKR50CO/DZIT1e+tj9Kw/uHLcMPO5hQqAsO64wXd+OplnNGZAzAg5H9Y6SrlYEQAAA=="""

def decode_embedded_text(encoded):
    compressed = base64.b64decode(encoded.encode("ascii"))
    return gzip.decompress(compressed).decode("utf-8")

train_text = decode_embedded_text(TRAIN_GZIP_B64)
test_text = decode_embedded_text(TEST_GZIP_B64)

print("Training file characters:", len(train_text))
print("Test file characters:", len(test_text))

Training file characters: 176855
Test file characters: 17504


In [ ]:
def parse_conll(text, source_name="dataset"):
    """Parse token<TAB>tag lines into a list of (tokens, tags) sentence pairs."""
    sentences = []
    tokens, tags = [], []

    for line_number, raw_line in enumerate(text.splitlines(), start=1):
        line = raw_line.strip("\r")
        if not line.strip():
            if tokens:
                sentences.append((tokens, tags))
                tokens, tags = [], []
            continue

        fields = line.split("\t")
        if len(fields) != 2:
            raise ValueError(
                f"{source_name}: expected TOKEN<TAB>TAG at line {line_number}, got {raw_line!r}"
            )

        token, tag = fields[0].strip(), fields[1].strip()
        if not token or not tag:
            raise ValueError(f"{source_name}: empty token/tag at line {line_number}")
        tokens.append(token)
        tags.append(tag)

    if tokens:
        sentences.append((tokens, tags))

    if not sentences:
        raise ValueError(f"{source_name}: no sentences parsed.")

    return sentences

train_sentences = parse_conll(train_text, "train_NP.txt")
test_sentences = parse_conll(test_text, "test_NP.txt")

tag_set = sorted({tag for _, tags in train_sentences for tag in tags})
print(f"Training sentences: {len(train_sentences):,}")
print(f"Test sentences: {len(test_sentences):,}")
print("Tags:", tag_set)
print("Example training sentence:")
print(list(zip(*train_sentences[0])))

Training sentences: 823
Test sentences: 77
Tags: ['B-NP', 'I-NP', 'O']
Example training sentence:
[('Rockwell', 'B-NP'), ('International', 'I-NP'), ('Corp.', 'I-NP'), ("'s", 'B-NP'), ('Tulsa', 'I-NP'), ('unit', 'I-NP'), ('said', 'O'), ('it', 'B-NP'), ('signed', 'O'), ('a', 'B-NP'), ('tentative', 'I-NP'), ('agreement', 'I-NP'), ('extending', 'O'), ('its', 'B-NP'), ('contract', 'I-NP'), ('with', 'O'), ('Boeing', 'B-NP'), ('Co.', 'I-NP'), ('to', 'O'), ('provide', 'O'), ('structural', 'B-NP'), ('parts', 'I-NP'), ('for', 'O'), ('Boeing', 'B-NP'), ("'s", 'B-NP'), ('747', 'I-NP'), ('jetliners', 'I-NP'), ('.', 'O')]


## 3. Estimate HMM probabilities with Add-\(k\) smoothing

An HMM uses:
- **Transition probabilities** \(P(t_i\mid t_{i-1})\)
- **Emission probabilities** \(P(w_i\mid t_i)\)
- Start and end transitions so the full sentence probability includes its boundary states.

We normalize words to lowercase for slightly better lexical coverage. Unknown test words are mapped to `<UNK>`. Add-\(k\) smoothing prevents unseen transitions/emissions from receiving zero probability. All calculations during decoding use natural logarithms.

In [ ]:
ADD_K = 0.1  # Add-k smoothing; a small positive value is suitable for this corpus.

def normalize_token(token):
    return token.lower()

class HMMTagger:
    def __init__(self, add_k=0.1):
        if add_k <= 0:
            raise ValueError("add_k must be positive.")
        self.add_k = float(add_k)
        self.tags = []
        self.vocab = set()
        self.tag_counts = Counter()
        self.emission_counts = Counter()
        self.transition_counts = Counter()
        self.outgoing_counts = Counter()
        self.start_counts = Counter()
        self.start_total = 0
        self.end_counts = Counter()
        self.end_total = 0
        self.transition_logp = {}
        self.emission_logp = {}

    def fit(self, sentences):
        self.tags = sorted({tag for _, tags in sentences for tag in tags})
        if not self.tags:
            raise ValueError("Training data has no tags.")

        self.tag_counts = Counter()
        self.emission_counts = Counter()
        self.transition_counts = Counter()
        self.outgoing_counts = Counter()
        self.start_counts = Counter()
        self.end_counts = Counter()
        self.start_total = 0
        self.end_total = 0
        self.vocab = set()

        for tokens, tags in sentences:
            if len(tokens) != len(tags) or not tokens:
                raise ValueError("Every sentence must have equally sized, non-empty token/tag lists.")

            words = [normalize_token(w) for w in tokens]
            self.vocab.update(words)
            self.start_counts[tags[0]] += 1
            self.start_total += 1

            previous = "<START>"
            for word, tag in zip(words, tags):
                self.tag_counts[tag] += 1
                self.emission_counts[(tag, word)] += 1
                self.transition_counts[(previous, tag)] += 1
                self.outgoing_counts[previous] += 1
                previous = tag

            self.transition_counts[(previous, "<END>")] += 1
            self.outgoing_counts[previous] += 1
            self.end_counts[previous] += 1
            self.end_total += 1

        self.vocab.add("<UNK>")
        self._build_log_probabilities()
        return self

    def _build_log_probabilities(self):
        k = self.add_k
        states_plus_end = self.tags + ["<END>"]
        # Transition distribution from START/tag includes all possible tags and END.
        previous_states = ["<START>"] + self.tags
        self.transition_logp = {}
        for prev in previous_states:
            denominator = self.outgoing_counts[prev] + k * len(states_plus_end)
            for nxt in states_plus_end:
                count = self.transition_counts[(prev, nxt)]
                self.transition_logp[(prev, nxt)] = math.log((count + k) / denominator)

        # Emission smoothing includes every training tag and the UNK bucket.
        vocab_size = len(self.vocab)
        self.emission_logp = {}
        for tag in self.tags:
            denominator = self.tag_counts[tag] + k * vocab_size
            for word in self.vocab:
                count = self.emission_counts[(tag, word)]
                self.emission_logp[(tag, word)] = math.log((count + k) / denominator)

    def _word(self, token):
        word = normalize_token(token)
        return word if word in self.vocab else "<UNK>"

    def emission_log_probability(self, word, tag):
        return self.emission_logp[(tag, self._word(word))]

    def forward_log_probability(self, tokens):
        """Return log P(tokens) using the forward algorithm, including END transition."""
        if not tokens:
            return self.transition_logp[("<START>", "<END>")] if ("<START>", "<END>") in self.transition_logp else -math.inf

        tags = self.tags
        first_word = tokens[0]
        alpha = {
            tag: self.transition_logp[("<START>", tag)] +
                 self.emission_log_probability(first_word, tag)
            for tag in tags
        }

        for word in tokens[1:]:
            next_alpha = {}
            for current_tag in tags:
                emit = self.emission_log_probability(word, current_tag)
                next_alpha[current_tag] = emit + logsumexp(
                    alpha[previous_tag] + self.transition_logp[(previous_tag, current_tag)]
                    for previous_tag in tags
                )
            alpha = next_alpha

        return logsumexp(
            alpha[tag] + self.transition_logp[(tag, "<END>")]
            for tag in tags
        )

    def viterbi(self, tokens):
        """Return the most likely tag sequence under the HMM, using log-space Viterbi."""
        if not tokens:
            return []

        tags = self.tags
        scores = {
            tag: self.transition_logp[("<START>", tag)] +
                 self.emission_log_probability(tokens[0], tag)
            for tag in tags
        }
        backpointers = []

        for word in tokens[1:]:
            next_scores = {}
            back = {}
            for current_tag in tags:
                candidates = {
                    previous_tag: scores[previous_tag] +
                                  self.transition_logp[(previous_tag, current_tag)]
                    for previous_tag in tags
                }
                best_previous = max(candidates, key=candidates.get)
                next_scores[current_tag] = (
                    candidates[best_previous] +
                    self.emission_log_probability(word, current_tag)
                )
                back[current_tag] = best_previous
            scores = next_scores
            backpointers.append(back)

        final_scores = {
            tag: scores[tag] + self.transition_logp[(tag, "<END>")]
            for tag in tags
        }
        last_tag = max(final_scores, key=final_scores.get)
        best_path = [last_tag]

        for back in reversed(backpointers):
            best_path.append(back[best_path[-1]])
        best_path.reverse()
        return best_path

def logsumexp(values):
    values = list(values)
    if not values:
        return -math.inf
    maximum = max(values)
    if maximum == -math.inf:
        return maximum
    return maximum + math.log(sum(math.exp(v - maximum) for v in values))

hmm = HMMTagger(add_k=ADD_K).fit(train_sentences)
print("HMM trained.")
print("Number of tags:", len(hmm.tags))
print("Vocabulary size (including <UNK>):", len(hmm.vocab))
print("Add-k:", ADD_K)

HMM trained.
Number of tags: 3
Vocabulary size (including <UNK>): 4256
Add-k: 0.1


## 4. Task 1 — Forward algorithm: probability of each test sentence
Probabilities of long sentences can be extremely small, so the notebook reports the natural-log probability and log10 probability. The raw probability is included when it is representable as a floating-point number; underflow is handled safely.

In [ ]:
def safe_probability_from_log(log_p):
    # Avoid Python overflow and make underflow explicit as 0.0.
    if log_p < -745:
        return 0.0
    if log_p > 709:
        return float("inf")
    return math.exp(log_p)

forward_rows = []
for sentence_id, (tokens, gold_tags) in enumerate(test_sentences, start=1):
    log_p = hmm.forward_log_probability(tokens)
    forward_rows.append({
        "sentence_id": sentence_id,
        "token_count": len(tokens),
        "log_probability": log_p,
        "log10_probability": log_p / math.log(10),
        "probability": safe_probability_from_log(log_p),
        "sentence": " ".join(tokens)
    })

forward_df = pd.DataFrame(forward_rows)
display(forward_df.head(10))
print("Test sentences with forward probabilities:", len(forward_df))
print("Log probability range:", forward_df["log_probability"].min(), "to", forward_df["log_probability"].max())

,sentence_id,token_count,log_probability,log10_probability,probability,sentence
0,1,37,-274.546060,-119.233839,5.836613e-120,Confidence in the pound is widely expected to ...
1,2,27,-200.696086,-87.161203,6.899178e-88,Chancellor of the Exchequer Nigel Lawson 's re...
2,3,29,-222.814930,-96.767295,1.708855e-97,But analysts reckon underlying support for ste...
3,4,36,-242.999843,-105.533491,2.927581e-106,This has increased the risk of the government ...
4,5,31,-226.682847,-98.447110,3.571827e-99,`` The risks for sterling of a bad trade figur...
5,6,36,-258.522330,-112.274822,5.311026e-113,"`` If there is another bad trade number , ther..."
6,7,39,-276.463726,-120.066671,8.576882e-121,"Forecasts for the trade figures range widely ,..."
7,8,25,-186.536715,-81.011866,9.730472e-82,The August deficit and the # 2.2 billion gap r...
8,9,28,-212.222537,-92.167077,6.806491e-93,"Sanjay Joshi , European economist at Baring Br..."
9,10,24,-174.540356,-75.801913,1.577926e-76,"At the same time , he remains fairly pessimist..."


Test sentences with forward probabilities: 77
Log probability range: -348.02484844106255 to -35.0053111949587


## 5. Task 2 — Viterbi decoding for every test sentence

In [ ]:
predicted_tags = [hmm.viterbi(tokens) for tokens, _ in test_sentences]
gold_tags = [tags for _, tags in test_sentences]
test_tokens = [tokens for tokens, _ in test_sentences]

assert len(predicted_tags) == len(gold_tags) == len(test_tokens)
for i, (tokens, predicted, gold) in enumerate(zip(test_tokens, predicted_tags, gold_tags)):
    assert len(tokens) == len(predicted) == len(gold), f"Token/tag length mismatch in sentence {i+1}"

print("Viterbi decoding completed for", len(predicted_tags), "sentences.")
print("\nExample predictions (first 3 sentences):")
for i in range(min(3, len(test_tokens))):
    print(f"\nSentence {i+1}:")
    print(f"{'TOKEN':<20} {'GOLD':<10} {'PREDICTED':<10}")
    for token, gold, pred in zip(test_tokens[i], gold_tags[i], predicted_tags[i]):
        print(f"{token:<20} {gold:<10} {pred:<10}")

Viterbi decoding completed for 77 sentences.

Example predictions (first 3 sentences):

Sentence 1:
TOKEN                GOLD       PREDICTED 
Confidence           B-NP       B-NP      
in                   O          O         
the                  B-NP       B-NP      
pound                I-NP       I-NP      
is                   O          O         
widely               O          B-NP      
expected             O          O         
to                   O          O         
take                 O          O         
another              B-NP       B-NP      
sharp                I-NP       I-NP      
dive                 I-NP       I-NP      
if                   O          O         
trade                B-NP       B-NP      
figures              I-NP       I-NP      
for                  O          O         
September            B-NP       B-NP      
,                    O          O         
due                  O          O         
for                  O          O       

## 6. Evaluate performance with `seqeval`
The notebook attempts to use `seqeval`, as suggested in the assignment. If the package is unavailable, it falls back to token-level metrics from scikit-learn, which is normally present in Colab. Entity-level NP precision/recall/F1 from seqeval is the preferred metric.

In [ ]:
seqeval_available = False
try:
    from seqeval.metrics import classification_report as seqeval_classification_report
    from seqeval.metrics import f1_score as seqeval_f1_score
    from seqeval.metrics import precision_score as seqeval_precision_score
    from seqeval.metrics import recall_score as seqeval_recall_score
    seqeval_available = True
except ImportError:
    print("seqeval is not installed. Attempting a quiet install...")
    import subprocess, sys
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "seqeval"],
        capture_output=True, text=True
    )
    if install_result.returncode == 0:
        try:
            from seqeval.metrics import classification_report as seqeval_classification_report
            from seqeval.metrics import f1_score as seqeval_f1_score
            from seqeval.metrics import precision_score as seqeval_precision_score
            from seqeval.metrics import recall_score as seqeval_recall_score
            seqeval_available = True
        except ImportError:
            seqeval_available = False

if seqeval_available:
    precision = seqeval_precision_score(gold_tags, predicted_tags, zero_division=0)
    recall = seqeval_recall_score(gold_tags, predicted_tags, zero_division=0)
    f1 = seqeval_f1_score(gold_tags, predicted_tags, zero_division=0)
    print("Entity-level metrics (seqeval):")
    print(seqeval_classification_report(gold_tags, predicted_tags, digits=4, zero_division=0))
    metric_type = "seqeval entity-level"
else:
    from sklearn.metrics import classification_report, accuracy_score
    flat_gold = [tag for seq in gold_tags for tag in seq]
    flat_pred = [tag for seq in predicted_tags for tag in seq]
    print("Token-level classification report (fallback):")
    print(classification_report(flat_gold, flat_pred, labels=hmm.tags, digits=4, zero_division=0))
    # Macro averages across token labels; report separately from entity-level seqeval scores.
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_gold, flat_pred, average="macro", zero_division=0
    )
    metric_type = "token-level macro (fallback)"

from sklearn.metrics import accuracy_score
flat_gold = [tag for seq in gold_tags for tag in seq]
flat_pred = [tag for seq in predicted_tags for tag in seq]
token_accuracy = accuracy_score(flat_gold, flat_pred)

metrics_df = pd.DataFrame([{
    "metric_type": metric_type,
    "precision": precision,
    "recall": recall,
    "F1": f1,
    "token_accuracy": token_accuracy,
    "test_sentences": len(test_sentences),
    "test_tokens": len(flat_gold)
}])
display(metrics_df)

seqeval is not installed. Attempting a quiet install...
Token-level classification report (fallback):
              precision    recall  f1-score   support

        B-NP     0.7715    0.8674    0.8167       475
        I-NP     0.8813    0.8463    0.8634       579
           O     0.9107    0.8717    0.8908       842

    accuracy                         0.8629      1896
   macro avg     0.8545    0.8618    0.8570      1896
weighted avg     0.8668    0.8629    0.8639      1896



,metric_type,precision,recall,F1,token_accuracy,test_sentences,test_tokens
0,token-level macro (fallback),0.8545,0.861796,0.856954,0.862869,77,1896


## 7. Save sentence probabilities, predictions, and metrics

In [ ]:
# Sentence-level output with gold and predicted tags stored as readable strings.
prediction_rows = []
for sentence_id, (tokens, gold, pred, forward_row) in enumerate(
    zip(test_tokens, gold_tags, predicted_tags, forward_rows), start=1
):
    prediction_rows.append({
        "sentence_id": sentence_id,
        "sentence": " ".join(tokens),
        "tokens": " | ".join(tokens),
        "gold_tags": " | ".join(gold),
        "predicted_tags": " | ".join(pred),
        "token_count": len(tokens),
        "log_probability": forward_row["log_probability"],
        "log10_probability": forward_row["log10_probability"],
        "probability": forward_row["probability"]
    })

predictions_df = pd.DataFrame(prediction_rows)
output_dir = os.path.join(os.getcwd(), "illm_lab7_outputs")
os.makedirs(output_dir, exist_ok=True)

forward_df.to_csv(os.path.join(output_dir, "forward_sentence_probabilities.csv"), index=False)
predictions_df.to_csv(os.path.join(output_dir, "viterbi_predictions_and_probabilities.csv"), index=False)
metrics_df.to_csv(os.path.join(output_dir, "evaluation_metrics.csv"), index=False)

print("Saved output files:")
for name in sorted(os.listdir(output_dir)):
    print("-", os.path.join(output_dir, name))
display(predictions_df.head(5))

Saved output files:
- /content/illm_lab7_outputs/evaluation_metrics.csv
- /content/illm_lab7_outputs/forward_sentence_probabilities.csv
- /content/illm_lab7_outputs/viterbi_predictions_and_probabilities.csv


,sentence_id,sentence,tokens,gold_tags,predicted_tags,token_count,log_probability,log10_probability,probability
0,1,Confidence in the pound is widely expected to ...,Confidence | in | the | pound | is | widely | ...,B-NP | O | B-NP | I-NP | O | O | O | O | O | B...,B-NP | O | B-NP | I-NP | O | B-NP | O | O | O ...,37,-274.546060,-119.233839,5.836613e-120
1,2,Chancellor of the Exchequer Nigel Lawson 's re...,Chancellor | of | the | Exchequer | Nigel | La...,O | O | B-NP | I-NP | O | O | B-NP | I-NP | I-...,B-NP | O | B-NP | I-NP | I-NP | O | B-NP | I-N...,27,-200.696086,-87.161203,6.899178e-88
2,3,But analysts reckon underlying support for ste...,But | analysts | reckon | underlying | support...,O | B-NP | O | B-NP | I-NP | O | B-NP | O | O ...,O | B-NP | I-NP | I-NP | I-NP | O | B-NP | O |...,29,-222.814930,-96.767295,1.708855e-97
3,4,This has increased the risk of the government ...,This | has | increased | the | risk | of | the...,B-NP | O | O | B-NP | I-NP | O | B-NP | I-NP |...,B-NP | O | O | B-NP | I-NP | O | B-NP | I-NP |...,36,-242.999843,-105.533491,2.927581e-106
4,5,`` The risks for sterling of a bad trade figur...,`` | The | risks | for | sterling | of | a | b...,O | B-NP | I-NP | O | B-NP | O | B-NP | I-NP |...,O | B-NP | I-NP | O | B-NP | O | B-NP | I-NP |...,31,-226.682847,-98.447110,3.571827e-99


## 8. Discussion and conclusion

The HMM is trained from token-level annotations by maximum-likelihood counts with Add-\(k\) smoothing. The forward algorithm sums probabilities over **all possible tag sequences** for a sentence, whereas Viterbi finds the **single most probable tag sequence**. Log probabilities are used because multiplying many small emission and transition probabilities would otherwise underflow.

Use the actual metrics printed above in the report:
- **Forward algorithm:** one sentence probability (plus log probability) for every test sentence.
- **Viterbi:** predicted `B-NP`, `I-NP`, and `O` tags for every token.
- **Evaluation:** entity-level precision, recall, and F1 with `seqeval` when available; token accuracy is also reported.

### Notes on interpretation
- `B-NP` marks the beginning of a noun phrase, `I-NP` marks continuation inside a noun phrase, and `O` marks tokens outside noun phrases.
- `seqeval` scores evaluate the extracted spans/entities; token accuracy measures individual tag correctness. These metrics are not interchangeable.
- The test data is used only for inference and evaluation, not for estimating HMM counts.